# tis script is to calcualte the absolute coordinates of bodyparts determined via DeepLabCut and the gradient at these coordinates

In [3]:
#Import packages
import pandas as pd
from pathlib import Path
import numpy as np
import os
import matplotlib.pyplot as plt
from natsort import natsorted
import glob
import matplotlib.patches as patches
import coordinate_conversion_functions as coord_conv
import curve_fitting_functions as curve_fit
import fnmatch
from collections import OrderedDict
from track_plotting_daniel import *

## centroid coordinates (output from micromanager)

In [18]:
path_center_coords='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_tracks/2020-07-01_13-21-00_chemotaxisl_worm1-TablePosRecord.txt'
center_coords=pd.read_csv(path_center_coords)
center_coords.head()

,time,xc,x,yc,y
0,13:21:00.988,x,21.3617,y,12.4125
1,13:21:05.079,x,21.6873,y,12.3068
2,13:21:05.095,x,21.6873,y,12.3068
3,13:21:05.095,x,21.6873,y,12.3068
4,13:21:05.095,x,21.6873,y,12.3068


## realtive coordinates of bodyparts determined via DLC

In [32]:
path_bodypart_coords='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/avi_all/2020-07-01_13-21-00_chemotaxisl_worm1-channel-0-bigtiffDLC_resnet50_HeadTailAug10shuffle1_275000_filtered.h5'
bodypart_coords=pd.read_hdf(path_bodypart_coords, low_memory=False)
bodypart_coords.head()

scorer    DLC_resnet50_HeadTailAug10shuffle1_275000                         \
bodyparts                                      Head                          
coords                                            x           y likelihood   
0                                        335.493225  279.720947   0.885174   
1                                        335.850616  291.608521   0.914306   
2                                        336.320496  291.608521   0.901548   
3                                        337.016968  291.608521   0.915392   
4                                        337.677704  291.608521   0.915135   

scorer                                        
bodyparts        Tail                         
coords              x           y likelihood  
0          322.490356  360.631958   0.928813  
1          323.039917  361.555389   0.775067  
2          323.086639  361.555389   0.816025  
3          323.304199  361.555389   0.818554  
4          323.348816  361.574158   0.838415

In [33]:
len(bodypart_coords)

200001

## get the absolute position of bodyparts

#### define parameters

In [20]:
px_mm=0.00325 # this value represents how much mm is one pixel
x_lenght_frame=608 #lenght of x axis in frame in recording used for DLC (measured with fiji)
y_width_frame=610 #width of the y axis in frame

#### calculate absolute head and tail positon

In [21]:
df_head_tail_center_coords=coord_conv.get_absolute_bodypart_coordinates(center_coords,bodypart_coords,px_mm,y_width_frame,x_lenght_frame)
df_head_tail_center_coords.head()

,absolute_x_Head,absolute_y_Head,absolute_x_Tail,absolute_y_Tail,x_center,y_center
0,21.259347,12.330343,21.301606,12.593304,21.3617,12.4125
1,21.583785,12.263278,21.625420,12.490605,21.6873,12.3068
2,21.582258,12.263278,21.625268,12.490605,21.6873,12.3068
3,21.579995,12.263278,21.624561,12.490605,21.6873,12.3068
4,21.577847,12.263278,21.624416,12.490666,21.6873,12.3068


## correct for food position
sets the coordinate of the food patch to 0
(x_ref measured manually after each recording)

In [22]:
#correct for food position
x_ref=8.7#9.10
df_head_tail_center_coords=coord_conv.adjust_for_food_position(df_head_tail_center_coords,x_ref)
df_head_tail_center_coords.head()

,absolute_x_Head,absolute_y_Head,absolute_x_Tail,absolute_y_Tail,x_center,y_center
0,12.559347,12.330343,12.601606,12.593304,12.6617,12.4125
1,12.883785,12.263278,12.925420,12.490605,12.9873,12.3068
2,12.882258,12.263278,12.925268,12.490605,12.9873,12.3068
3,12.879995,12.263278,12.924561,12.490605,12.9873,12.3068
4,12.877847,12.263278,12.924416,12.490666,12.9873,12.3068


## calculate gradient exposure of bodyparts
(parameters obtained by fitting a curve to bromophenol diffusion on agar (4h)--> see curve_fit script)

In [23]:
#quadratic
a=0.00675494
b=-0.16437685
c=1

In [24]:
#exponential
y0=1         
plateau=0
K=0.23754334

In [25]:
#sigmoid
L=1
x0=3.81953189
k=-0.69421567
b=0

In [26]:
df_head_tail_center_coords=coord_conv.calculate_concentration_for_bodyparts(df_head_tail_center_coords.copy(),'sigmoid',L,x0,k,b)
df_head_tail_center_coords.head()

,absolute_x_Head,absolute_y_Head,absolute_x_Tail,absolute_y_Tail,x_center,y_center,concentration_absolute_x_Head,concentration_absolute_x_Tail,concentration_x_center
0,12.559347,12.330343,12.601606,12.593304,12.6617,12.4125,0.002312,0.002245,0.002154
1,12.883785,12.263278,12.925420,12.490605,12.9873,12.3068,0.001847,0.001794,0.001719
2,12.882258,12.263278,12.925268,12.490605,12.9873,12.3068,0.001849,0.001794,0.001719
3,12.879995,12.263278,12.924561,12.490605,12.9873,12.3068,0.001851,0.001795,0.001719
4,12.877847,12.263278,12.924416,12.490666,12.9873,12.3068,0.001854,0.001795,0.001719


## calculate concentration change from one frame to the next

In [27]:
#add columns with difference in concentration for all bodyparts

#get concentration columns
all_bodyparts_concentration=df_head_tail_center_coords[df_head_tail_center_coords.columns[pd.Series(df_head_tail_center_coords.columns).str.contains('concentration')]]

#add new columns with change in concentration
for current_bodypart in all_bodyparts_concentration.columns:
        df_head_tail_center_coords[f'change_in_{current_bodypart}']=all_bodyparts_concentration[current_bodypart].diff()

## adding a column representing time passed (in seconds)

In [28]:
#add a column displaying time
fps=167 #frames per second
df_head_tail_center_coords['seconds']=np.arange(0, len(df_head_tail_center_coords)/fps,1/fps)
df_head_tail_center_coords

,absolute_x_Head,absolute_y_Head,absolute_x_Tail,absolute_y_Tail,x_center,y_center,concentration_absolute_x_Head,concentration_absolute_x_Tail,concentration_x_center,change_in_concentration_absolute_x_Head,change_in_concentration_absolute_x_Tail,change_in_concentration_x_center,seconds
0,12.559347,12.330343,12.601606,12.593304,12.6617,12.4125,0.002312,0.002245,0.002154,NaN,NaN,NaN,0.000000
1,12.883785,12.263278,12.925420,12.490605,12.9873,12.3068,0.001847,0.001794,0.001719,-0.000465,-4.512226e-04,-0.000435,0.005988
2,12.882258,12.263278,12.925268,12.490605,12.9873,12.3068,0.001849,0.001794,0.001719,0.000002,1.887970e-07,0.000000,0.011976
3,12.879995,12.263278,12.924561,12.490605,12.9873,12.3068,0.001851,0.001795,0.001719,0.000003,8.793819e-07,0.000000,0.017964
4,12.877847,12.263278,12.924416,12.490666,12.9873,12.3068,0.001854,0.001795,0.001719,0.000003,1.803949e-07,0.000000,0.023952
...,...,...,...,...,...,...,...,...,...,...,...,...,...
199997,0.488273,16.671920,0.702163,16.455699,0.3021,16.3923,0.909916,0.896981,0.919964,-0.000138,0.000000e+00,0.000000,1197.586826
199998,0.492050,16.671920,0.702063,16.455100,0.3020,16.3923,0.909701,0.896988,0.919969,-0.000215,6.414774e-06,0.000005,1197.592814
199999,0.493533,16.668119,0.701963,16.455100,0.3019,16.3923,0.909616,0.896994,0.919974,-0.000085,6.414420e-06,0.000005,1197.598802
200000,0.496709,16.667919,0.701863,16.455000,0.3018,16.3922,0.909435,0.897001,0.919979,-0.000181,6.414067e-06,0.000005,1197.604790


## save dataframe as csv

In [30]:
#save as csv
output_path='/groups/zimmer/Daniel_Mitic/data/'
filename='2020-07-01_13-21-00_chemotaxisl_worm1.csv'
print(output_path+filename)
df_head_tail_center_coords.to_csv(output_path+filename)

/groups/zimmer/Daniel_Mitic/data/2020-07-01_13-21-00_chemotaxisl_worm1.csv


## all the steps above but for all recordings

### define paths and read in csv containing x_refs for all recordings as a column

In [31]:
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/raw_data/'
output_path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/body_part_coordinates/'
all_x_ref=pd.read_csv('/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/raw_data/all_x_refs.csv')
all_x_ref.head()

,name,x_ref
0,2020-07-01_10-10-48_control_worm1-TablePosReco...,7.64
1,2020-07-01_10-50-18_control_worm2-TablePosReco...,7.64
2,2020-07-01_11-50-53_control_worm3-TablePosReco...,7.64
3,2020-07-01_13-21-00_chemotaxisl_worm1-TablePos...,9.00
4,2020-07-01_14-41-11_chemotaxisl_worm2-TablePos...,10.50


### define parameters needed for calculating absolute coordinates of the bodyparts

In [ ]:

px_mm=0.00325 # this value represents how much mm is one pixel total_lenght_mm/total_lengt_pixel
x_lenght_frame=608 #lenght of x axis in frame (measured with fiji)
y_width_frame=610 #width of the y axis in frame
fps=167

### loop to create csv with absolute coordinates of bodyparts + x_ref correction

In [34]:


for i, folder in enumerate(natsorted(os.listdir(path))):
    
    #stop the loop when last x_ref has been reached
    if i>len(all_x_ref)-1:break
    
    #get current x_ref
    x_ref=all_x_ref['x_ref'][i] 
    print(i, ': x ref of ', folder, 'is: ', x_ref)
    
    #get bodypart coordinates from DeepLabCut
    hdf5_files=glob.glob(os.path.join(path,folder,'*.h5'))
    hdf5_file=hdf5_files[0]
    bodypart_coords=pd.read_hdf(hdf5_file)
    
    #get centroid coordinates from micromanager
    txt_files=glob.glob(os.path.join(path,folder,'*.txt'))
    txt_file=txt_files[0]
    center_coords=pd.read_csv(txt_file)
   
    
    #some center coordinates have more rows than dlc data. this checks and adjusts the lenght of the centroid coordinates 
    if len(center_coords) > len(bodypart_coords):
        rows_diff=len(center_coords) - len(bodypart_coords)
        print(txt_files,'is',rows_diff,' rows longer')
        center_coords.drop(center_coords.tail(rows_diff).index,inplace=True)
    center_coords = center_coords[0:len(bodypart_coords)] 
    
    
    
    #get absolute coordinates for the bodyparts
    df_head_tail_center_coords=coord_conv.get_absolute_bodypart_coordinates(center_coords,bodypart_coords,px_mm,y_width_frame,x_lenght_frame)
    
    #adjust for food position
    df_head_tail_center_coords=coord_conv.adjust_for_food_position(df_head_tail_center_coords,x_ref)
    
    #add column with time passed
    df_head_tail_center_coords['seconds']=np.arange(0, len(df_head_tail_center_coords)/fps,1/fps)
    
    #save as csv
    #df_head_tail_center_coords.to_csv(output_path+folder+'_bodypart_coordinates.csv')

0 : x ref of  2020-07-01_10-10-48_control_worm1-TablePosRecord is:  7.639999999999999
['/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/raw_data/2020-07-01_10-10-48_control_worm1-TablePosRecord/2020-07-01_10-10-48_control_worm1-TablePosRecord.txt'] is 1  rows longer
        absolute_x_Head  absolute_y_Head  absolute_x_Tail  absolute_y_Tail  \
0             13.962351        24.376240        14.905561        23.993702   
1             13.862682        24.426580        14.801465        24.028304   
2             13.862595        24.426580        14.799063        24.028875   
3             13.862499        24.426580        14.799063        24.028987   
4             13.861733        24.426580        14.799063        24.029305   
...                 ...              ...              ...              ...   
199996        24.163553        10.062531        24.393371         9.274600   
199997        24.163453        10.062551        24.393419         9.275000   
199998        24.163253     

## define input and output paths
input--> csv containing absolute coordinates of bodyparts

In [4]:
path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/body_part_coordinates/'
output_path='/groups/zimmer/Daniel_Mitic/data/ulises_chemotaxis/'

### define parameters for calculating the gradient

In [5]:
#this is telling you the concentration of every x position occuring in the data
#parameters determined via the curve fit script
L=1
x0=3.81953189
k=-0.69421567
b=0

### loop to add concentration and concentration change for all bodyparts

In [7]:
for filename in natsorted(os.listdir(path)):
    if filename.endswith('.csv'):
        bodypart_coords=pd.read_csv(path+filename)
        
        #calculate concentration
        bodypart_coords=coord_conv.calculate_concentration_for_bodyparts(bodypart_coords.copy(),'sigmoid',L,x0,k,b)
        
        #calculate concentration change
        all_bodyparts_concentration=bodypart_coords[bodypart_coords.columns[pd.Series(bodypart_coords.columns).str.contains('concentration')]]

        #add new columns with change in concentration
        for current_bodypart in all_bodyparts_concentration.columns:
            bodypart_coords[f'change_in_{current_bodypart}']=all_bodyparts_concentration[current_bodypart].diff() 
        
        #for some reason a second index column is added  which i delete here
        bodypart_coords.drop(bodypart_coords.columns[0], axis=1, inplace=True)
     
    #save as csv
    bodypart_coords.to_csv(output_path+filename+'_bodypart_coordinates_concentration.csv',index=False)
    
